In [1]:
import os
import glob
import pandas as pd

files = sorted(glob.glob('../data/raw/*.csv'))
datasets = {}

for f in files:
    name = os.path.basename(f).replace('.csv', '')

    if name.startswith('nav_'):
        continue

    short_name = name.split('-', 2)[-1]

    try:
        datasets[short_name] = pd.read_csv(f)
    except UnicodeDecodeError:
        datasets[short_name] = pd.read_csv(f, encoding='ISO-8859-1')

print(len(datasets))
print(list(datasets.keys()))

10
['08_investor_transactions', '09_portfolio_holdings', '10_benchmark_indices', '01_fund_master', '02_nav_history', '03_aum_by_fund_house', '04_monthly_sip_inflows', '05_category_inflows', '06_industry_folio_count', '07_scheme_performance']


In [2]:
master = datasets['01_fund_master']
nav = datasets['02_nav_history']

print('Fund master:', master.shape)
print('NAV history:', nav.shape)

Fund master: (40, 15)
NAV history: (46000, 3)


In [3]:
print('Type in fund master:', master['amfi_code'].dtype)
print('Type in NAV history:', nav['amfi_code'].dtype)
print('Empty codes in fund master:', master['amfi_code'].isnull().sum())
print('Empty codes in NAV history:', nav['amfi_code'].isnull().sum())

Type in fund master: int64
Type in NAV history: int64
Empty codes in fund master: 0
Empty codes in NAV history: 0


In [4]:
master_codes = set(master['amfi_code'].dropna())
nav_codes = set(nav['amfi_code'].dropna())

print('Unique codes in fund master:', len(master_codes))
print('Unique codes in NAV history:', len(nav_codes))

Unique codes in fund master: 40
Unique codes in NAV history: 40


In [5]:
missing_in_nav = master_codes - nav_codes

print('Codes in fund master but not in NAV history:', len(missing_in_nav))
print(sorted(missing_in_nav)[:20])

if len(missing_in_nav) == 0:
    print('PASS: every fund master code is in NAV history.')
else:
    print('FAIL: some fund master codes are not in NAV history.')

Codes in fund master but not in NAV history: 0
[]
PASS: every fund master code is in NAV history.


In [6]:
master[master['amfi_code'].isin(missing_in_nav)][['amfi_code', 'scheme_name', 'fund_house']]

,amfi_code,scheme_name,fund_house


In [7]:
extra_in_nav = nav_codes - master_codes

print('Codes in NAV history but not in fund master:', len(extra_in_nav))
print(sorted(extra_in_nav)[:20])

Codes in NAV history but not in fund master: 0
[]


In [8]:
n_repeated_master = master['amfi_code'].duplicated().sum()
print('Repeated codes in fund master:', n_repeated_master)

Repeated codes in fund master: 0


In [9]:
rows_per_code = nav.groupby('amfi_code').size()

print(rows_per_code.describe())
print()
print(rows_per_code.sort_values().head(10))

count      40.0
mean     1150.0
std         0.0
min      1150.0
25%      1150.0
50%      1150.0
75%      1150.0
max      1150.0
dtype: float64

amfi_code
100016    1150
100025    1150
100033    1150
101206    1150
101207    1150
101208    1150
102885    1150
102886    1150
102887    1150
118632    1150
dtype: int64


In [10]:
print('Missing values in each column:')
print(nav.isnull().sum())
print()

n_duplicate_rows = nav.duplicated().sum()
n_repeated_pairs = nav.duplicated(subset=['amfi_code', 'date']).sum()
print('Duplicate rows:', n_duplicate_rows)
print('Repeated (amfi_code, date) pairs:', n_repeated_pairs)

nav_values = pd.to_numeric(nav['nav'], errors='coerce')
n_bad_nav = nav_values.isnull().sum() - nav['nav'].isnull().sum()
n_nav_not_positive = (nav_values <= 0).sum()
print('NAV values that are not numbers:', n_bad_nav)
print('NAV values that are zero or negative:', n_nav_not_positive)

nav_dates = pd.to_datetime(nav['date'], errors='coerce')
n_bad_dates = nav_dates.isnull().sum() - nav['date'].isnull().sum()
print('Dates that are not valid:', n_bad_dates)
print('Oldest date:', nav_dates.min())
print('Newest date:', nav_dates.max())

Missing values in each column:
amfi_code    0
date         0
nav          0
dtype: int64

Duplicate rows: 0
Repeated (amfi_code, date) pairs: 0
NAV values that are not numbers: 0
NAV values that are zero or negative: 0
Dates that are not valid: 0
Oldest date: 2022-01-03 00:00:00
Newest date: 2026-05-29 00:00:00


In [11]:
result_text = 'PASS' if len(missing_in_nav) == 0 else 'FAIL'

lines = [
    '# Data quality summary: AMFI codes',
    '',
    f'- Fund master: {len(master)} rows, {len(master_codes)} unique AMFI codes, {n_repeated_master} repeated codes.',
    f'- NAV history: {len(nav)} rows, {len(nav_codes)} unique AMFI codes.',
    f'- Check 1 ({result_text}): {len(missing_in_nav)} fund master codes are not in NAV history.',
    f'- Check 2: {len(extra_in_nav)} NAV history codes are not in the fund master.',
    f'- NAV rows for each code: smallest {rows_per_code.min()}, largest {rows_per_code.max()}.',
    f'- NAV date range: {nav_dates.min()} to {nav_dates.max()}.',
    f'- NAV table problems: {nav.isnull().sum().sum()} empty values, {n_duplicate_rows} duplicate rows, {n_repeated_pairs} repeated (code, date) pairs.',
    f'- NAV values: {n_bad_nav} not numbers, {n_nav_not_positive} zero or negative. Dates not valid: {n_bad_dates}.',
]

text = '\n'.join(lines)
print(text)

# Data quality summary: AMFI codes

- Fund master: 40 rows, 40 unique AMFI codes, 0 repeated codes.
- NAV history: 46000 rows, 40 unique AMFI codes.
- Check 1 (PASS): 0 fund master codes are not in NAV history.
- Check 2: 0 NAV history codes are not in the fund master.
- NAV rows for each code: smallest 1150, largest 1150.
- NAV date range: 2022-01-03 00:00:00 to 2026-05-29 00:00:00.
- NAV table problems: 0 empty values, 0 duplicate rows, 0 repeated (code, date) pairs.
- NAV values: 0 not numbers, 0 zero or negative. Dates not valid: 0.


In [12]:
with open('../reports/data_quality_amfi_codes.md', 'w', encoding='utf-8') as f:
    f.write(text)

print('Saved')

Saved


Check 1 passed. All fund master codes have NAV history.

No action needed
